# FXS / 13 — Hidden Markov regimes on daily returns

Pre-registration: `discussions/FX_Intraday_Seasonality/008_claude_fxs13_hmm_regimes_preregistration.md`.

- **The model.** A Gaussian HMM (K = 2, 3) per pair, on the daily New York-day return and the log realized volatility (5-minute). States are ordered by volatility.
- **Walk-forward.** For each year 2021–2026, the HMM is fitted on 2016 → previous year. The state used on day t is the **one-step prediction from the close of t−1** (forward filter only, no look-ahead).

**The tests:**
- **H1:** does the predicted state change the pair's next-day return?
- **H2:** does trading each of the 12 signals only in its historically good states beat trading it every day? Compared with no filter and with C0 (the signal's own past).
- **H3:** the 12-signal portfolio with the HMM filter, $6,000, 1–10×.

§0 also fits the HMM on 2016–2025 and plots the **smoothed** states. That plot is description only: it uses the future.

In [ ]:
import json, os, sys, tempfile, time
import datetime as dt
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from IPython.display import display

E0 = 6_000.0
LEVERAGES = [1, 2, 3, 5, 10]
COMMISSION_PER_LOT = 3.0
SPREAD_FACTOR = 0.9
MIN_BUCKET = 30
KS = [2, 3]
TEST_YEARS = list(range(2021, 2027))
HMM_WINDOWS = ((dt.date(2016, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
               (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
SIG_WINDOWS = ((dt.date(2020, 1, 1), dt.date(2020, 12, 31), "discovery"), (dt.date(2021, 1, 1), dt.date(2025, 12, 31), "confirmation"),
               (dt.date(2026, 1, 1), dt.date(2026, 11, 6), "holdout"))
SIGNALS = {
    "EURUSD-02": ("EURUSD", "NY", 660, 900, -1), "USDJPY-01": ("USDJPY", "UTC", 55, 115, -1), "USDJPY-02": ("USDJPY", "UTC", 25, 55, 1),
    "N-EU-1": ("EURUSD", "NY", 885, 900, -1), "N-UJ-1": ("USDJPY", "UTC", 55, 70, -1), "N-UJ-3": ("USDJPY", "NY", 1340, 1400, 1),
    "N-EJ-1": ("EURJPY", "NY", 1315, 1435, 1), "N-EJ-2": ("EURJPY", "NY", 780, 900, -1),
    "AUDJPY-N1": ("AUDJPY", "NY", 1195, 1435, 1), "AUDJPY-N2": ("AUDJPY", "UTC", 60, 180, 1), "CADJPY-N1": ("CADJPY", "NY", 1070, 1310, 1),
    "GBPJPY-N1": ("GBPJPY", "NY", 1195, 1435, 1)}

SYNTHETIC = os.environ.get("FXS_SYNTHETIC") == "1"
if SYNTHETIC:
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("FXS_SYNTH_DIR") or tempfile.mkdtemp(prefix="fxs13_")
    os.environ.pop("DUKASCOPY_DIR", None)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "equity_trading").exists())
sys.path.insert(0, str(ROOT / "src"))
from equity_trading import fx_seasonality as F
from equity_trading.hmm import GaussianHMM

cfg = F.FXSConfig()
OUT = Path.cwd() / ("13_synthetic" if SYNTHETIC else "13"); OUT.mkdir(exist_ok=True)
cand = pd.DataFrame([{"id": k, "pair": p, "clock": c, "start": s, "end": e, "direction": d} for k, (p, c, s, e, d) in SIGNALS.items()])
PAIRS = sorted(set(cand["pair"]))
PAIR_OF = cand.set_index("id")["pair"]
pd.set_option("display.width", 240, "display.max_columns", 40, "display.max_rows", 200)
np.seterr(invalid="ignore", divide="ignore")

In [ ]:
if SYNTHETIC:
    base = Path(os.environ["EQUITY_TRADING_DIR"]) / "Dukascopy"
    for i, p in enumerate(PAIRS):
        F.write_synthetic_candles(base, p, dt.date(2015, 12, 28), dt.date(2026, 11, 6), seed=101 + i, drift_bps=-3.0 if p == "EURUSD" else 0.0,
                                  drift_ny=(4, 8), price=150.0 if p.endswith("JPY") else 1.10)

## 1. Daily features (2016 → 2026) and the signals' daily net results (2020 → 2026)

In [ ]:
FEAT, RET = {}, {}
for p in PAIRS:
    parts = []
    for a, b, st in HMM_WINDOWS:
        pn = F.load_panels([p], a, b, st, cfg, clocks=("NY",))[(p, "NY")]
        lm = np.log(pn.mid)
        r = (lm[:, -1] - lm[:, 0]) * 1e4
        rv = np.sqrt(np.nansum(np.diff(lm[:, ::5], axis=1) ** 2, axis=1)) * 1e4
        parts.append(pd.DataFrame({"ret_bps": r, "log_rv": np.log(rv), "close": pn.mid[:, -1]}, index=pd.to_datetime(pn.dates)))
    f = pd.concat(parts).sort_index()
    f = f[~f.index.duplicated()].replace([np.inf, -np.inf], np.nan).dropna()
    FEAT[p] = f
    RET[p] = F.bps_to_pips(f["ret_bps"], f["close"], p)          # the day's move in pips
    print(p, len(f), "days")

G, P = [], []
for a, b, st in SIG_WINDOWS:
    g, _, pr = F.structure_detail(cand, a, b, st, cfg, with_price=True, spread_factor=SPREAD_FACTOR)
    G.append(g); P.append(pr)
G = pd.concat(G).sort_index().fillna(0.0)
P = pd.concat(P).sort_index().ffill().bfill()
comm = pd.Series({i: F.commission_bps_per_lot(PAIR_OF[i], COMMISSION_PER_LOT) for i in G.columns})
traded = G != 0
NB = (G - comm).where(traded, 0.0)
pips = pd.DataFrame({i: F.bps_to_pips(NB[i], P[i], PAIR_OF[i]) for i in G.columns}, index=G.index).where(traded)
idx = G.index

## 2. Walk-forward HMM states (one-step predictions, causal)

In [ ]:
def fit_year(p, K, Y):
    f = FEAT[p]
    tr = f[f.index.year < Y][["ret_bps", "log_rv"]]
    mu, sd = tr.mean(), tr.std()
    h = GaussianHMM(K).fit(((tr - mu) / sd).to_numpy(), n_restarts=3, n_iter=150, seed=Y)
    Z = ((f[["ret_bps", "log_rv"]] - mu) / sd).to_numpy()
    pred = h.predict_next(Z)                                   # row t: P(state of the NEXT day | data to close t)
    nxt = pd.DataFrame(pred, index=f.index).shift(1)           # now row t = prediction made at the close of t-1
    return h, nxt

STATE, PARAMS = {}, []
t0 = time.time()
for p in PAIRS:
    for K in KS:
        s = pd.Series(np.nan, index=FEAT[p].index)
        for Y in TEST_YEARS:
            h, nxt = fit_year(p, K, Y)
            sel = nxt.index.year == Y
            s[sel] = nxt[sel].to_numpy().argmax(1)
            for k in range(K):
                PARAMS.append({"pair": p, "K": K, "fit through": Y - 1, "state": k, "persistence": h.A[k, k],
                               "mean ret (z)": h.mu[k, 0], "mean log-vol (z)": h.mu[k, 1], "sd ret (z)": np.sqrt(h.var[k, 0])})
        STATE[(p, K)] = s
    print(p, f"{time.time() - t0:.0f}s")
PARAMS = pd.DataFrame(PARAMS)
display(PARAMS[PARAMS["fit through"] == 2025].round(3)); PARAMS.to_csv(OUT / "13_hmm_params.csv", index=False)
occ = pd.DataFrame({f"{p} K{K}": s.dropna().value_counts(normalize=True).sort_index() for (p, K), s in STATE.items()}).T
print("share of 2021-2026 days in each predicted state:"); display(occ.round(2))

## H1. Does the predicted state move the pair's next-day return?

Mean daily move (pips) and realized volatility in each predicted state, 2021–2026. Volatility should differ (states are volatility regimes); the question is the **mean**.

In [ ]:
h1 = []
for (p, K), s in STATE.items():
    d = pd.DataFrame({"state": s, "ret": RET[p], "rv": np.exp(FEAT[p]["log_rv"])}).dropna()
    groups = [g["ret"].to_numpy() for _, g in d.groupby("state")]
    row = {"pair": p, "K": K, "ANOVA p (mean)": stats.f_oneway(*groups).pvalue if len(groups) > 1 else np.nan}
    for k, g in d.groupby("state"):
        row[f"s{int(k)} days"] = len(g); row[f"s{int(k)} mean pips"] = g["ret"].mean()
        row[f"s{int(k)} t"] = g["ret"].mean() / (g["ret"].std() / np.sqrt(len(g))); row[f"s{int(k)} vol bps"] = g["rv"].mean()
    h1.append(row)
h1 = pd.DataFrame(h1)
display(h1.round(3)); h1.to_csv(OUT / "13_H1_next_day.csv", index=False)

## H2. The 12 signals traded only in their good states (walk-forward)

In [ ]:
def state_for(sig, K):
    return STATE[(PAIR_OF[sig], K)].reindex(idx)

def mask(sig, filt):
    x = pips[sig]
    take = pd.Series(False, index=idx)
    for Y in TEST_YEARS:
        tr, te = (idx.year < Y) & (idx.year >= 2020), idx.year == Y
        if filt == "C0":
            take[te] = x[tr].mean() > 0
            continue
        K = int(filt[-1])
        s = state_for(sig, K)
        # training-day states: each year 2021+ carries its own walk-forward state; 2020 carries the states of the
        # fit through 2020 (HMM parameters in sample for that one year; the signal outcomes are not used by the fit)
        allow = []
        for k in range(K):
            xs = x[tr & (s == k).to_numpy()].dropna()
            if len(xs) < MIN_BUCKET or xs.mean() > 0:
                allow.append(k)
        take[te] = np.isin(s[te].to_numpy(), allow)
    return take & traded[sig]

# 2020 has no walk-forward state (the first fit is through 2020); give 2020 the states of the fit through 2020
for (p, K) in STATE:
    h, nxt = fit_year(p, K, 2021)
    sel = nxt.index.year == 2020
    STATE[(p, K)][sel] = nxt[sel].to_numpy().argmax(1)

FILTERS = ["C0", "HMM2", "HMM3"]
oos = idx.year >= TEST_YEARS[0]
MASK = {(i, f): mask(i, f) for i in SIGNALS for f in FILTERS}
rows = []
for i in SIGNALS:
    base = pips[i].fillna(0.0)
    c0 = base.where(MASK[(i, "C0")], 0.0)
    for f in FILTERS:
        fl = base.where(MASK[(i, f)], 0.0)
        m, se, t = F.nw_tstats((fl - base)[oos].to_numpy()[:, None], 5)
        m0, se0, t0 = F.nw_tstats((fl - c0)[oos].to_numpy()[:, None], 5)
        rows.append({"signal": i, "filter": f, "share traded": MASK[(i, f)][oos].sum() / traded[i][oos].sum(),
                     "pips/day unfiltered": base[oos].mean(), "pips/day filtered": fl[oos].mean(), "diff": m[0], "t diff": t[0],
                     "p": 2 * stats.norm.sf(abs(t[0])), "t vs C0": t0[0], "2026 filtered": fl[idx.year == 2026].mean(),
                     "2026 unfiltered": base[idx.year == 2026].mean()})
H2 = pd.DataFrame(rows)
cm = H2["filter"] != "C0"
H2.loc[cm, "q"] = F.bh_reject(H2.loc[cm, "p"].to_numpy(), 0.05)[1]
H2["usable"] = (H2["q"] <= 0.05) & (H2["diff"] > 0) & (H2["t vs C0"] > 0) & (H2["2026 filtered"] > 0)
H2.to_csv(OUT / "13_H2_signals.csv", index=False)
print("diff = filtered - unfiltered, net pips per day, 2021-2026:"); display(H2.pivot(index="signal", columns="filter", values="diff").round(3))
print("t:"); display(H2.pivot(index="signal", columns="filter", values="t diff").round(2))
display(H2.round(3))
print("usable:"); display(H2[H2["usable"]])

# per-state results of each signal, 2021-2026 (states predicted out of sample)
ps = []
for i in SIGNALS:
    for K in KS:
        s = state_for(i, K)
        for k in range(K):
            x = pips[i][oos & (s == k).to_numpy()].dropna()
            ps.append({"signal": i, "K": K, "state": k, "trades": len(x), "net pips": x.mean(), "t": x.mean() / (x.std() / np.sqrt(len(x))) if len(x) > 2 else np.nan})
ps = pd.DataFrame(ps)
display(ps.pivot_table(index="signal", columns=["K", "state"], values="net pips").round(2)); ps.to_csv(OUT / "13_signal_by_state.csv", index=False)

## H3. The 12-signal portfolio with the HMM filters

In [ ]:
rowsC, curves = [], {}
for f in ["none", *FILTERS]:
    Nf = NB.copy() if f == "none" else NB.where(pd.DataFrame({i: MASK[(i, f)] for i in SIGNALS}), 0.0)
    for per, sel in (("2021-2026", oos), ("2026", idx.year == 2026)):
        for L in LEVERAGES:
            acc = F.simulate_account(Nf.loc[sel], pd.Series(float(L), index=Nf.columns), E0)
            st = F.account_stats(acc, E0)
            if per == "2021-2026":
                curves[(f, L)] = acc
            rowsC.append({"filter": f, "period": per, "L": L, "final_equity": round(st["final_equity"]), "cagr %": round(100 * st["cagr"], 1),
                          "sharpe": round(st["sharpe"], 2), "max dd %": round(100 * st["max_drawdown"], 1), "worst year %": round(100 * st["worst_year"], 1)})
Ct = pd.DataFrame(rowsC); Ct.to_csv(OUT / "13_H3_portfolio.csv", index=False)
display(Ct.pivot_table(index="filter", columns=["period", "L"], values="cagr %"))
display(Ct.pivot_table(index="filter", columns=["period", "L"], values="max dd %"))
fig, ax = plt.subplots(figsize=(13, 5))
for f in ["none", *FILTERS]:
    ax.plot(curves[(f, 2)]["equity"], label=f)
ax.set_yscale("log"); ax.axhline(E0, color="grey", lw=0.6); ax.legend(); ax.set_title("12 signals, 2x each")
plt.tight_layout(); plt.savefig(OUT / "13_equity_2x.png", dpi=110); plt.show()

## Description only: smoothed states, HMM fitted on 2016–2025 (uses the future)

In [ ]:
fig, axes = plt.subplots(len(PAIRS), 1, figsize=(15, 2.6 * len(PAIRS)), sharex=True)
for ax, p in zip(axes, PAIRS):
    f = FEAT[p][FEAT[p].index.year <= 2025]
    mu, sd = f[["ret_bps", "log_rv"]].mean(), f[["ret_bps", "log_rv"]].std()
    h = GaussianHMM(3).fit(((f[["ret_bps", "log_rv"]] - mu) / sd).to_numpy(), n_restarts=3, n_iter=150)
    st = h.filter(((f[["ret_bps", "log_rv"]] - mu) / sd).to_numpy()).argmax(1)
    ax.plot(f.index, f["close"], color="black", lw=0.6)
    for k, col in zip(range(3), ("tab:green", "tab:orange", "tab:red")):
        ax.fill_between(f.index, f["close"].min(), f["close"].max(), where=st == k, color=col, alpha=0.18, step="mid", label=f"state {k}")
    ax.set_title(f"{p}: filtered 3-state HMM fitted on 2016-2025 (states ordered calm -> volatile)"); ax.legend(fontsize=7, loc="upper left")
plt.tight_layout(); plt.savefig(OUT / "13_states.png", dpi=100); plt.show()

In [ ]:
summary = {"study": "FXS13", "prereg": "discussions/FX_Intraday_Seasonality/008_claude_fxs13_hmm_regimes_preregistration.md", "synthetic": SYNTHETIC,
           "H1": h1.round(4).to_dict(orient="records"), "H2_usable": H2[H2["usable"]].round(4).to_dict(orient="records"),
           "H2": H2.round(4).to_dict(orient="records"), "H3_2x": Ct[Ct["L"] == 2].to_dict(orient="records")}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2, default=str))
print("written", OUT / "summary.json")